# MAG7 Multi-Model Options Backtest

This notebook runs the AAPL research pipeline across the Mag7 universe:
- a trade selector on price plus ratio/metrics features,
- a pairwise learn-to-rank option selector,
- a multi-leg mean-variance basket selector trained from hybrid labels.

It backtests each symbol independently from 2025 onward, then aggregates the per-symbol summaries.

Run the quant-warehouse backfill step first if the local ThetaData ArcticDB store is incomplete. The research pass itself reads warehouse data.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

repo_root = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / 'quant_orchestrator').is_dir()
)
warehouse_root = repo_root.parent / 'quant-warehouse'
if not (warehouse_root / 'quant_warehouse').is_dir():
    raise RuntimeError(f'Could not find quant-warehouse next to {repo_root}')

for path in (repo_root, warehouse_root):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from quant_orchestrator.options_research import (
    MAG7_SYMBOLS,
    maybe_backfill_options,
    run_mag7_options_research,
)

ARTIFACT_DIR = repo_root / 'artifacts' / 'mag7_options_multi_model_backtest'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'repo_root={repo_root}')
print(f'warehouse_root={warehouse_root}')
print(f'artifact_dir={ARTIFACT_DIR}')
print(f'mag7_symbols={MAG7_SYMBOLS}')

In [ ]:
SYMBOLS = MAG7_SYMBOLS
TRADE_START = '2018-01-01'
TRADE_END = None
TRAIN_CUTOFF = '2025-01-01'
DOWNLOAD_MISSING = False
RUN_BACKFILL = False
BACKFILL_START = '2024-01-01'
BACKFILL_END = pd.Timestamp.today().normalize().date().isoformat()
MAX_DTE = 90
STRIKE_RANGE = 12
TARGET_TENOR_DAYS = 60
K_PARAMS = {'M': [1], 'QE': [1], 'YE': list(range(1, 13))}

if RUN_BACKFILL:
    backfill_summary = maybe_backfill_options(
        symbols=SYMBOLS,
        start_date=BACKFILL_START,
        end_date=BACKFILL_END,
        source='arctic-fmp',
        overwrite=False,
        skip_existing=True,
    )
    print(json.dumps(backfill_summary, indent=2, default=str))
else:
    print('Skipping live backfill. Populate ThetaData ArcticDB with quant-warehouse first, then rerun research from the warehouse.')

In [ ]:
results = run_mag7_options_research(
    symbols=SYMBOLS,
    trade_start=TRADE_START,
    trade_end=TRADE_END,
    train_cutoff=TRAIN_CUTOFF,
    download_missing=DOWNLOAD_MISSING,
    max_dte=MAX_DTE,
    strike_range=STRIKE_RANGE,
    target_tenor_days=TARGET_TENOR_DAYS,
    k_params=K_PARAMS,
)

summary_frame = results['summary_frame']
display(summary_frame)

In [ ]:
successful = []
for item in results['symbol_results']:
    if 'curve_summary' in item:
        curve = item['curve_summary'].copy()
        curve.insert(0, 'symbol', item['symbol'])
        successful.append(curve)

curve_frame = pd.concat(successful, ignore_index=True) if successful else pd.DataFrame()
if not curve_frame.empty:
    display(curve_frame)

print('successful_symbols', int(summary_frame['error'].isna().sum()) if 'error' in summary_frame.columns else len(summary_frame))

In [ ]:
summary_frame.to_parquet(ARTIFACT_DIR / 'mag7_summary_frame.parquet', index=False)
summary_frame.to_csv(ARTIFACT_DIR / 'mag7_summary_frame.csv', index=False)
if not curve_frame.empty:
    curve_frame.to_parquet(ARTIFACT_DIR / 'mag7_curve_summary.parquet', index=False)
    curve_frame.to_csv(ARTIFACT_DIR / 'mag7_curve_summary.csv', index=False)

with open(ARTIFACT_DIR / 'run_metadata.json', 'w') as f:
    json.dump(
        {
            'symbols': list(SYMBOLS),
            'trade_start': TRADE_START,
            'trade_end': TRADE_END,
            'train_cutoff': TRAIN_CUTOFF,
            'download_missing': DOWNLOAD_MISSING,
            'run_backfill': RUN_BACKFILL,
            'max_dte': MAX_DTE,
            'strike_range': STRIKE_RANGE,
            'target_tenor_days': TARGET_TENOR_DAYS,
            'k_params': K_PARAMS,
        },
        f,
        indent=2,
        default=str,
    )

print('saved artifacts to', ARTIFACT_DIR)